# HARP Ablation Study on Kaggle
#
Upload `kaggle_harp_ablation.zip` as a Kaggle Dataset, attach it to this
notebook, then run the cells below. The zip already contains the source code,
dataset CSVs, and final model artifacts needed by `src.analytics.ablation.ablation_study`.

In [1]:
import sys
import subprocess

packages = [
    "gymnasium",
    "stable-baselines3[extra]",
    "sb3-contrib",
    "shimmy>=0.2.1",
    "matplotlib",
    "pandas",
    "networkx",
    "scipy",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 10.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 2.0 MB/s eta 0:00:00
Dependencies installed


In [2]:
import os
import shutil
import zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/CoreRouter")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

def looks_like_harp_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "src" / "analytics" / "ablation" / "ablation_study.py").exists()
        and (path / "src" / "orchestration" / "jo_vdpr" / "env.py").exists()
    )


def zip_contains_harp_package(path: Path) -> bool:
    if not path.is_file() or path.suffix.lower() != ".zip":
        return False
    try:
        with zipfile.ZipFile(path, "r") as archive:
            names = set(archive.namelist())
            return (
                "src/analytics/ablation/ablation_study.py" in names
                and "src/orchestration/jo_vdpr/env.py" in names
            )
    except zipfile.BadZipFile:
        return False


zip_candidates = [
    p for p in list(KAGGLE_INPUT.glob("**/*.zip")) + list(Path("/kaggle/working").glob("*.zip"))
    if zip_contains_harp_package(p)
]

if zip_candidates:
    package_path = zip_candidates[0]
    print(f"Using zip package: {package_path}")
    with zipfile.ZipFile(package_path, "r") as archive:
        archive.extractall(WORK_DIR)
else:
    # Kaggle often auto-extracts uploaded datasets. Dataset names are arbitrary,
    # so detect the package by its file structure instead of by dataset name.
    dir_candidates = [
        p for p in KAGGLE_INPUT.glob("**")
        if looks_like_harp_root(p)
    ]
    dir_candidates += [
        p.parent.parent.parent for p in KAGGLE_INPUT.glob("**/src/analytics/ablation/ablation_study.py")
        if looks_like_harp_root(p.parent.parent.parent)
    ]
    dir_candidates = list(dict.fromkeys(dir_candidates))

    if not dir_candidates:
        raise FileNotFoundError(
            "Cannot find HARP ablation package under /kaggle/input. "
            "Attach any Kaggle Dataset that contains src/analytics/ablation/ablation_study.py, "
            "or upload any zip containing that project structure."
        )

    package_path = dir_candidates[0]
    print(f"Using extracted dataset directory: {package_path}")
    for item in package_path.iterdir():
        dest = WORK_DIR / item.name
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)

    # If the user attached an old notebook output, it might be in a different directory
    # Let's search for any 'results' folder in KAGGLE_INPUT and merge it to WORK_DIR/results
    print("\nScanning for previous results/models in Kaggle inputs...")
    for results_dir in KAGGLE_INPUT.rglob("results"):
        if results_dir.is_dir() and "models" in [d.name for d in results_dir.iterdir()]:
            print(f"Found results directory at: {results_dir}")
            # Merge contents of this results dir into WORK_DIR/results
            def merge_dirs(src: Path, dst: Path):
                dst.mkdir(parents=True, exist_ok=True)
                for item in src.iterdir():
                    s = item
                    d = dst / item.name
                    if s.is_dir():
                        merge_dirs(s, d)
                    elif not d.exists():
                        shutil.copy2(s, d)
            merge_dirs(results_dir, WORK_DIR / "results")
            print(f"Merged {results_dir} into {WORK_DIR / 'results'}")

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

def ensure_sb3_model_zip(topology: str) -> None:
    """Kaggle may auto-extract nested Stable-Baselines3 model zip files.

    The project loader expects results/models/v11/dgrl_v11_final_<topology>.zip
    to be a real zip file containing policy.pth. If Kaggle extracted it into a
    directory, re-pack that directory back into the expected archive name.
    """
    model_dir = WORK_DIR / "results" / "models" / "v11"
    expected_zip = model_dir / f"dgrl_v11_final_{topology}.zip"

    if expected_zip.is_file():
        return

    candidates = [
        expected_zip,
        model_dir / f"dgrl_v11_final_{topology}",
        model_dir / f"dgrl_v11_{topology}",
        model_dir / topology,
    ]
    candidates += [
        p for p in model_dir.rglob("*")
        if p.is_dir() and p.name.startswith(f"dgrl_v11") and topology in p.name
    ]
    candidates += [
        p.parent for p in model_dir.rglob("policy.pth")
        if topology in str(p.parent)
    ]

    source_dir = next((p for p in candidates if p.is_dir() and (p / "policy.pth").exists()), None)
    if source_dir is None:
        source_dir = next((p for p in candidates if p.is_dir() and list(p.rglob("policy.pth"))), None)
    if source_dir is None:
        raise FileNotFoundError(
            f"Cannot find extracted Stable-Baselines3 model directory for topology={topology}"
        )

    if expected_zip.exists() and expected_zip.is_dir():
        moved_dir = expected_zip.with_name(expected_zip.name + "_extracted")
        if moved_dir.exists():
            shutil.rmtree(moved_dir)
        shutil.move(str(expected_zip), str(moved_dir))
        source_dir = moved_dir

    print(f"Repacking extracted model for {topology}: {source_dir} -> {expected_zip}")
    with zipfile.ZipFile(expected_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        policy_files = list(source_dir.rglob("policy.pth"))
        root = policy_files[0].parent if policy_files else source_dir
        for file_path in root.rglob("*"):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(root))

for topo in ["vietnam", "nsfnet", "geant2"]:
    try:
        ensure_sb3_model_zip(topo)
    except FileNotFoundError:
        print(f"Skipping model check for {topo} (file/folder not found)")

for required in [
    "src/analytics/ablation/ablation_study.py",
    "src/analytics/training/train_harp_ablation.py",
    "src/orchestration/jo_vdpr/env.py",
    "data/real_telecom_combined.csv",
]:
    assert Path(required).exists(), f"Missing required file: {required}"

print(f"Extracted to {WORK_DIR}")

Using extracted dataset directory: /kaggle/input/datasets/gamemasterhh/v21-harp/CoreRouter

Scanning for previous results/models in Kaggle inputs...
Found results directory at: /kaggle/input/datasets/gamemasterhh/v21-harp/CoreRouter/results
Merged /kaggle/input/datasets/gamemasterhh/v21-harp/CoreRouter/results into /kaggle/working/CoreRouter/results
Repacking extracted model for vietnam: /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_vietnam/dgrl_v11_final_vietnam -> /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_vietnam.zip
Repacking extracted model for nsfnet: /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_nsfnet -> /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_nsfnet.zip
Repacking extracted model for geant2: /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_geant2 -> /kaggle/working/CoreRouter/results/models/v11/dgrl_v11_final_geant2.zip
Extracted to /kaggle/working/CoreRouter


In [3]:
from pathlib import Path

# =====================================================================
# CHANGE THIS LINE FOR EACH KAGGLE ACCOUNT: "vietnam", "nsfnet", or "geant2"
# =====================================================================
WORKER_TOPOLOGY = "nsfnet"

STEPS = 3000
SEED = 42

# Define optimal steps based on topology size (action space)
_STEPS_MAP = {
    "vietnam": 500_000,   # 10 nodes -> 100 actions
    "nsfnet": 2_000_000,  # 14 nodes -> 196 actions
    "geant2": 2_000_000,  # 23 nodes -> 529 actions
}
TRAIN_TOTAL_STEPS = _STEPS_MAP.get(WORKER_TOPOLOGY, 500_000)
TRAIN_N_ENVS = 4

# Scenarios to evaluate
WORKER_SCENARIOS = [
    ("uniform", "uniform"),
    ("bursty", "bursty"),
    ("heavyTail", "heavy_tail"),
]

WORKER_ROOT = Path("/kaggle/working/harp_ablation") / f"worker_{WORKER_TOPOLOGY}"
WORKER_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Worker topology: {WORKER_TOPOLOGY}")
print(f"Train steps per variant: {TRAIN_TOTAL_STEPS}")
print(f"Eval steps per scenario: {STEPS}")
print(f"Scenarios: {[label for label, _ in WORKER_SCENARIOS]}")

Worker topology: nsfnet
Train steps per variant: 2000000
Eval steps per scenario: 3000
Scenarios: ['uniform', 'bursty', 'heavyTail']


In [4]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = f"{WORK_DIR}:{env.get('PYTHONPATH', '')}"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MPLCONFIGDIR"] = "/tmp/matplotlib"

train_cmd = [
    sys.executable,
    "-m",
    "src.analytics.training.train_harp_ablation",
    "--variant", "all",
    "--topology", WORKER_TOPOLOGY,
    "--scenario", "heavy_tail", # Train on hardest scenario
    "--total-steps", str(TRAIN_TOTAL_STEPS),
    "--n-envs", str(TRAIN_N_ENVS),
    "--seed", str(SEED),
    "--data-path", "data/real_telecom_combined.csv",
    "--output-root", "results/models/ablation",
]

# ─── Check if we already have trained models AND they are compatible ───────────
import zipfile, io, torch

def _check_model_compatible(model_zip: Path, expected_action_dim: int) -> bool:
    """Return True only if the model's action_net output dim matches the env."""
    if not model_zip.exists():
        return False
    try:
        with zipfile.ZipFile(model_zip) as z:
            pth_names = [n for n in z.namelist() if n.endswith("policy.pth")]
            if not pth_names:
                return False
            with z.open(pth_names[0]) as f:
                sd = torch.load(io.BytesIO(f.read()), map_location="cpu",
                                weights_only=False)
        w = sd.get("action_net.weight")
        if w is None:
            return True   # Can't determine – assume OK
        return int(w.shape[0]) == expected_action_dim
    except Exception as e:
        print(f"  Warning: could not inspect model {model_zip}: {e}")
        return False

def _compute_action_dim(topology: str) -> int:
    from src.orchestration.jo_vdpr.topology import TopologyManager
    tm = TopologyManager(topology)
    return tm.num_nodes * tm.num_nodes  # Discrete(N*N)


expected_dim = _compute_action_dim(WORKER_TOPOLOGY)
model_out = Path("results/models/ablation")
existing_zips = list(model_out.glob("**/*.zip")) if model_out.exists() else []

# Check if harp_full variant (used for load sweep) is compatible
harp_full_zip = model_out / "harp_full" / f"dgrl_v11_harp_full_{WORKER_TOPOLOGY}.zip"
all_compatible = (
    len(existing_zips) >= 5
    and _check_model_compatible(harp_full_zip, expected_dim)
)

if all_compatible:
    print(f"✅ Ablation models exist and are compatible (action_dim={expected_dim}). Skipping training.")
else:
    if existing_zips and not all_compatible:
        print(f"⚠️  Found existing models but action_dim mismatch (expected={expected_dim}). Retraining...")
    print("======================================================")
    print(f"Training ablation checkpoints for {WORKER_TOPOLOGY}...")
    print("======================================================")
    print(" ".join(train_cmd))
    subprocess.check_call(train_cmd, env=env)
    print("Training completed.")

Training ablation checkpoints for nsfnet...
/usr/bin/python3 -m src.analytics.training.train_harp_ablation --variant all --topology nsfnet --scenario heavy_tail --total-steps 2000000 --n-envs 4 --seed 42 --data-path data/real_telecom_combined.csv --output-root results/models/ablation


2026-06-17 14:06:23.294703: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781705183.695061      81 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781705183.817437      81 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781705184.851423      81 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781705184.851475      81 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781705184.851478      81 computation_placer.cc:177] computation placer alr

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | 3.22e+03 |
| time/              |          |
|    fps             | 439      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 100        |
|    ep_rew_mean          | 3.14e+03   |
| time/                   |            |
|    fps                  | 412        |
|    iterations           | 2          |
|    time_elapsed         | 9          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01666606 |
|    clip_fraction        | 0.192      |
|    clip_range           | 0.2        |
|    entropy_loss         | -4.69      |
|    explained_variance   | 0.196      |
|    learning_rate        |

2026-06-17 14:07:25,435 [INFO] step=8192 sla_viol=2.17% lambda=-100.1


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | 3.15e+03    |
| time/                   |             |
|    fps                  | 432         |
|    iterations           | 4           |
|    time_elapsed         | 18          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.025602989 |
|    clip_fraction        | 0.23        |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.6        |
|    explained_variance   | 0.93        |
|    learning_rate        | 0.000299    |
|    loss                 | -0.309      |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.0649     |
|    value_loss           | 0.0289      |
-----------------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 100     

2026-06-17 14:07:43,398 [INFO] step=16384 sla_viol=2.16% lambda=-100.1


... (1,955 outputs left out)

2026-06-17 18:42:37,697 [INFO] Saved results/models/ablation/harp_soft_msd_penalty/dgrl_v11_harp_soft_msd_penalty_nsfnet.zip
2026-06-17 18:42:37,697 [INFO] Saved results/models/ablation/harp_soft_msd_penalty/vec_normalize_v11_harp_soft_msd_penalty_nsfnet.pkl
2026-06-17 18:42:41.678016: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-17 18:42:41.678016: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-17 18:42:41.694407: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-17 18:42:41.705940: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register c

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | 4.19e+03 |
| time/              |          |
|    fps             | 616      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | 4.09e+03    |
| time/                   |             |
|    fps                  | 517         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.016601365 |
|    clip_fraction        | 0.202       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.69       |
|    explained_variance   | 0.194       |
|    learnin

2026-06-17 19:57:50,212 [INFO] Saved results/models/ablation/harp_wo_adaptive_penalty/dgrl_v11_harp_wo_adaptive_penalty_nsfnet.zip
2026-06-17 19:57:50,213 [INFO] Saved results/models/ablation/harp_wo_adaptive_penalty/vec_normalize_v11_harp_wo_adaptive_penalty_nsfnet.pkl


Training completed.


In [5]:
import pandas as pd
import shutil
from IPython.display import display

SEEDS = [42, 100, 2024, 8888, 9999, 7, 17, 123, 2025, 31415]
summary_dfs = []

for scenario_label, env_scenario in WORKER_SCENARIOS:
    for seed in SEEDS:
        out_dir = WORKER_ROOT / f"{WORKER_TOPOLOGY}_{scenario_label}_{STEPS}_seed_{seed}"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        eval_cmd = [
            sys.executable,
            "-m",
            "src.analytics.ablation.ablation_study",
            "--topology", WORKER_TOPOLOGY,
            "--scenario", env_scenario,
            "--steps", str(STEPS),
            "--seed", str(seed),
            "--version", "v11",
            "--data-path", "data/real_telecom_combined.csv",
            "--model-root", "results/models",
            "--output-dir", str(out_dir),
        ]
        
        print(f"\nEvaluating topology={WORKER_TOPOLOGY}, scenario={scenario_label}, seed={seed}")
        try:
            subprocess.check_call(eval_cmd, env=env)
        except subprocess.CalledProcessError:
            print(f"⚠️ Evaluation failed for {scenario_label} (seed {seed}). Check logs.")
            continue

        summary_path = out_dir / "harp_ablation_summary.csv"
        if summary_path.exists():
            df_tmp = pd.read_csv(summary_path)
            df_tmp.insert(0, "Topology", WORKER_TOPOLOGY)
            df_tmp.insert(1, "Scenario", scenario_label)
            # Make sure seed is tracked
            df_tmp["seed"] = seed
            summary_dfs.append(df_tmp)

if not summary_dfs:
    raise RuntimeError("No summary files were produced!")

# Merge into a single master CSV for all seeds
raw_summary = pd.concat(summary_dfs, ignore_index=True)

# Group by variant, topology, scenario and calculate mean and std
# Only use numeric cols that actually exist in the CSV
all_possible_numeric = ["acceptance_rate", "safe_acceptance_rate", "attempted_msd_violation_rate",
                        "admitted_msd_violation_rate", "sla_violation_rate", "evacuation_hit_rate",
                        "switching_rate", "avg_latency_ms", "avg_reward"]
numeric_cols = [c for c in all_possible_numeric if c in raw_summary.columns]
group_cols = [g for g in ["variant", "Topology", "Scenario"] if g in raw_summary.columns]

print(f"Aggregating columns: {numeric_cols}")
mean_df = raw_summary.groupby(group_cols)[numeric_cols].mean().reset_index()
std_df  = raw_summary.groupby(group_cols)[numeric_cols].std().reset_index()

# Format as "mean ± std"
final_summary = mean_df.copy()
for col in numeric_cols:
    final_summary[col] = (mean_df[col].round(3).astype(str)
                          + " \u00b1 "
                          + std_df[col].round(3).astype(str))

# Also save the raw per-seed CSV for transparency
raw_summary.to_csv(WORKER_ROOT / f"{WORKER_TOPOLOGY}_raw_ablation_10seeds.csv", index=False)

final_summary_path = WORKER_ROOT / f"{WORKER_TOPOLOGY}_master_ablation_summary_10seeds.csv"
final_summary.to_csv(final_summary_path, index=False)

print("\n======================================================")
print("FINAL ABLATION SUMMARY (10 SEEDS)")
print("======================================================")
display(final_summary)


Evaluating topology=nsfnet, scenario=uniform, seed=42


2026-06-17 19:58:00.276362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781726280.300008     179 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781726280.308302     179 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781726280.329872     179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726280.329901     179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726280.329904     179 computation_placer.cc:177] computation placer alr


Variant                              |  Accept |    Safe | MSD try | MSD admit |     SLA |  Latency |   Reward
HARP full                            |   31.6% |   31.6% |   68.4% |      0.0% |    0.1% |    2.63 |     13.1
HARP w/o GAT                         |   26.6% |   26.6% |   73.4% |      0.0% |    0.4% |    9.88 |      9.2
HARP w/o hard masking                |   33.3% |   33.3% |   65.5% |      0.0% |    0.0% |    5.94 |     17.9
HARP w/o mask + unchecked admission  |   48.1% |   10.0% |   87.8% |     38.1% |    0.0% |    7.88 |     28.2
HARP with soft MSD penalty           |   48.6% |    5.9% |   90.2% |     42.8% |    0.0% |    1.43 |      8.0
HARP w/o adaptive penalty            |   31.9% |   31.9% |   68.1% |      0.0% |    0.1% |    5.16 |     19.9

Evaluating topology=nsfnet, scenario=uniform, seed=100


2026-06-17 19:59:03.063453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781726343.086190     183 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781726343.093668     183 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781726343.113137     183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726343.113161     183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726343.113164     183 computation_placer.cc:177] computation placer alr


Variant                              |  Accept |    Safe | MSD try | MSD admit |     SLA |  Latency |   Reward
HARP full                            |   30.3% |   30.3% |   69.7% |      0.0% |    0.1% |    2.74 |     12.6
HARP w/o GAT                         |   27.1% |   27.1% |   72.9% |      0.0% |    0.5% |   10.14 |      9.9
HARP w/o hard masking                |   32.8% |   32.8% |   66.4% |      0.0% |    0.0% |    5.70 |     17.3
HARP w/o mask + unchecked admission  |   47.3% |    9.9% |   89.0% |     37.4% |    0.0% |    7.76 |     28.0
HARP with soft MSD penalty           |   47.8% |    7.3% |   86.4% |     40.5% |    0.0% |    1.40 |      8.1
HARP w/o adaptive penalty            |   32.1% |   32.1% |   67.8% |      0.0% |    0.0% |    3.76 |     19.9

Evaluating topology=nsfnet, scenario=uniform, seed=2024


2026-06-17 20:00:03.188772: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781726403.211392     187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781726403.219108     187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781726403.238820     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726403.238844     187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781726403.238846     187 computation_placer.cc:177] computation placer alr

... (22 outputs left out)


Variant                              |  Accept |    Safe | MSD try | MSD admit |     SLA |  Latency |   Reward
HARP full                            |   29.3% |   29.3% |   70.6% |      0.0% |    0.0% |    3.86 |     12.1
HARP w/o GAT                         |   25.9% |   25.9% |   74.0% |      0.0% |    0.9% |   10.60 |      8.9
HARP w/o hard masking                |   32.6% |   32.6% |   66.7% |      0.0% |    0.3% |    6.75 |     17.1
HARP w/o mask + unchecked admission  |   44.2% |   11.1% |   85.1% |     33.1% |    0.6% |    9.77 |     26.3
HARP with soft MSD penalty           |   45.6% |    5.5% |   91.6% |     40.1% |    0.0% |    1.40 |      8.2
HARP w/o adaptive penalty            |   30.7% |   30.7% |   69.3% |      0.0% |    0.0% |    4.28 |     18.5

Evaluating topology=nsfnet, scenario=heavyTail, seed=9999


2026-06-17 20:12:12.442350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781727132.465225     235 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781727132.472748     235 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781727132.492674     235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781727132.492699     235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781727132.492702     235 computation_placer.cc:177] computation placer alr


Variant                              |  Accept |    Safe | MSD try | MSD admit |     SLA |  Latency |   Reward
HARP full                            |   29.2% |   29.2% |   70.8% |      0.0% |    0.0% |    3.39 |     11.9
HARP w/o GAT                         |   27.1% |   27.1% |   72.9% |      0.0% |    0.8% |   10.24 |     10.3
HARP w/o hard masking                |   32.1% |   32.1% |   67.4% |      0.0% |    0.4% |    7.39 |     16.6
HARP w/o mask + unchecked admission  |   45.3% |   10.2% |   87.0% |     35.2% |    0.8% |   10.08 |     27.2
HARP with soft MSD penalty           |   46.3% |    6.1% |   91.3% |     40.2% |    0.0% |    1.19 |      8.2
HARP w/o adaptive penalty            |   30.1% |   30.1% |   69.8% |      0.0% |    0.1% |    5.06 |     17.9
Aggregating columns: ['acceptance_rate', 'safe_acceptance_rate', 'attempted_msd_violation_rate', 'admitted_msd_violation_rate', 'sla_violation_rate', 'evacuation_hit_rate', 'switching_rate', 'avg_latency_ms', 'avg_reward']

FINA

,variant,Topology,Scenario,acceptance_rate,safe_acceptance_rate,attempted_msd_violation_rate,admitted_msd_violation_rate,sla_violation_rate,evacuation_hit_rate,switching_rate,avg_latency_ms,avg_reward
0,HARP full,nsfnet,bursty,31.513 ± 0.806,31.513 ± 0.806,68.48 ± 0.801,0.0 ± 0.0,0.007 ± 0.015,15.493 ± 3.058,54.533 ± 3.248,2.04 ± 0.724,15.483 ± 0.766
1,HARP full,nsfnet,heavyTail,29.413 ± 0.472,29.413 ± 0.472,70.553 ± 0.471,0.0 ± 0.0,0.047 ± 0.056,23.133 ± 6.312,63.02 ± 1.88,3.556 ± 0.353,11.832 ± 0.363
2,HARP full,nsfnet,uniform,31.347 ± 0.605,31.347 ± 0.605,68.653 ± 0.605,0.0 ± 0.0,0.1 ± 0.062,20.68 ± 3.736,58.473 ± 1.083,2.58 ± 0.284,13.482 ± 0.66
3,HARP w/o GAT,nsfnet,bursty,27.153 ± 1.068,27.153 ± 1.068,72.827 ± 1.066,0.0 ± 0.0,0.433 ± 0.097,9.707 ± 2.785,89.9 ± 0.836,9.96 ± 0.33,10.791 ± 1.054
4,HARP w/o GAT,nsfnet,heavyTail,26.88 ± 0.615,26.88 ± 0.615,73.093 ± 0.593,0.0 ± 0.0,0.893 ± 0.119,26.713 ± 5.686,92.787 ± 0.563,10.342 ± 0.186,9.656 ± 0.611
5,HARP w/o GAT,nsfnet,uniform,27.46 ± 1.107,27.46 ± 1.107,72.527 ± 1.124,0.0 ± 0.0,0.593 ± 0.196,26.853 ± 3.858,88.427 ± 1.455,9.948 ± 0.299,10.066 ± 0.965
6,HARP w/o adaptive penalty,nsfnet,bursty,33.667 ± 0.609,33.667 ± 0.609,66.28 ± 0.623,0.0 ± 0.0,0.0 ± 0.0,15.5 ± 7.331,52.887 ± 1.152,1.315 ± 0.273,21.506 ± 0.752
7,HARP w/o adaptive penalty,nsfnet,heavyTail,30.613 ± 0.572,30.613 ± 0.572,69.347 ± 0.562,0.0 ± 0.0,0.013 ± 0.03,50.733 ± 10.812,53.647 ± 1.46,4.24 ± 0.49,18.379 ± 0.614
8,HARP w/o adaptive penalty,nsfnet,uniform,32.753 ± 0.684,32.753 ± 0.684,67.18 ± 0.72,0.0 ± 0.0,0.113 ± 0.104,47.973 ± 11.645,55.68 ± 2.532,4.575 ± 0.525,20.802 ± 0.846
9,HARP w/o hard masking,nsfnet,bursty,33.313 ± 0.849,33.313 ± 0.849,66.24 ± 0.809,0.0 ± 0.0,0.0 ± 0.0,3.96 ± 3.179,81.493 ± 0.467,4.595 ± 0.127,18.162 ± 0.996


In [8]:
print("\n======================================================")
print("ZIPPING RESULTS FOR DOWNLOAD")
print("======================================================")

# Move load sweep results to worker root
# run_load_sweep.py saves to results/benchmark_algorithm/load_sweep/
load_sweep_out = Path("results/benchmark_algorithm/load_sweep")
if not load_sweep_out.exists():
    load_sweep_out = Path("results/load_sweep")  # legacy fallback
if load_sweep_out.exists():
    dest_dir = WORKER_ROOT / "load_sweep_results"
    dest_dir.mkdir(parents=True, exist_ok=True)
    for f in load_sweep_out.rglob("*"):
        if f.is_file():
            shutil.copy2(f, dest_dir / f.name)
    print(f"Copied {len(list(dest_dir.glob('*')))} load sweep files to {dest_dir}")
else:
    print("⚠️ Load sweep output not found — may have failed earlier.")

archive_base = Path("/kaggle/working") / f"harp_ablation_results_{WORKER_TOPOLOGY}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=WORKER_ROOT)

print(f"\n✅ All done for {WORKER_TOPOLOGY}!")
print(f"Master CSV: {final_summary_path}")
print(f"Download Archive: {archive_path}")


ZIPPING RESULTS FOR DOWNLOAD
Copied 1 load sweep files to /kaggle/working/harp_ablation/worker_nsfnet/load_sweep_results

✅ All done for nsfnet!
Master CSV: /kaggle/working/harp_ablation/worker_nsfnet/nsfnet_master_ablation_summary_5seeds.csv
Download Archive: /kaggle/working/harp_ablation_results_nsfnet.zip


In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
from pathlib import Path

manifest = {
    "topology": WORKER_TOPOLOGY,
    "evaluation_steps": STEPS,
    "evaluation_seeds": SEEDS,
    "scenarios": WORKER_SCENARIOS,
    "python": sys.version,
    "platform": platform.platform(),
}
try:
    manifest["git_commit"] = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORK_DIR, text=True).strip()
except Exception:
    manifest["git_commit"] = None

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest["input_sha256"] = sha256_file(WORK_DIR / "data/real_telecom_combined.csv")
manifest["checkpoints"] = {}
for path in sorted((WORK_DIR / "results/models/ablation").glob("**/*.zip")):
    manifest["checkpoints"][str(path.relative_to(WORK_DIR))] = sha256_file(path)
(WORKER_ROOT / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

archive_base = Path("/kaggle/working") / f"harp_ablation_results_{WORKER_TOPOLOGY}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=WORKER_ROOT)
print(f"Ablation results archive: {archive_path}")
